In [1]:
import os
import math
import json
import numpy as np
import random

# 런팟
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset
from sklearn.metrics import f1_score

In [2]:
# Config
SEARCH = True   # True=레시피 탐색(짧은 런) / False=최종 풀런

config = {
    "num_labels": 188,
    "seed": 42,
    "learning_rate": 1.2e-4,
    "epochs": 2 if SEARCH else 12,
    "early_stop": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 512,          
    "eff_batch": 128,           
    "micro_batch": 128,
    "eval_micro_batch": 512,
    "repo_final": "ingyoun/A.X-patent-maxlen512-batch",
    "out_path": "/workspace/output/modernbert-maxlen512_batch",
    "tag": "modernbert-patent-len512",
    "rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
}

config["grad_accum"] = config["eff_batch"] // config["micro_batch"]   # micro×accum = eff_batch
config["run_name"] = (
    f"axenc_len512_eff{config['eff_batch']}_lr{config['learning_rate']:.0e}"
    + ("_search" if SEARCH else "_full")
)   # 조합별로 wandb 곡선·출력이 겹치지 않게 (eff_batch, lr, 모드)를 이름에 박음

In [3]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [4]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


## 데이터셋

In [5]:
dataset = load_dataset(
    "ingyoun/patent-clean-text-modernbert-tokenized",
    cache_dir="/workspace/hf_cache",
)

dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

## 모델

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"], 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5,
        dtype=torch.float32,
        attn_implementation="flash_attention_2",            # len8192와 구현 일치
    )

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 토크나이저

In [7]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=config["rev"])
EOS_ID = tokenizer.eos_token_id


def _prep(batch):
    max_len = config["max_len"]
    ids, masks = [], []
    for x, m in zip(batch["input_ids"], batch["attention_mask"]):
        if len(x) > max_len:
            x = x[: max_len - 1] + [EOS_ID]   # <s> 유지 + 꼬리를 <\s>로 마감
            m = m[:max_len]
        ids.append(x)
        masks.append(m)
    return {"input_ids": ids, "attention_mask": masks, "length": [len(i) for i in ids]}


dataset = dataset.map(_prep, batched=True)
print(f"EOS_ID={EOS_ID}  max_len={config['max_len']}")
dataset

Map:   0%|          | 0/201895 [00:00<?, ? examples/s]

Map:   0%|          | 0/11271 [00:00<?, ? examples/s]

Map:   0%|          | 0/11162 [00:00<?, ? examples/s]

EOS_ID=1  max_len=512


DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11162
    })
})

## 커스텀

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: int = 2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [9]:
class FocalTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.focal = FocalLoss(0.25, 2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        loss = self.focal(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [10]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    
    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [11]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    """
    sigmoid→τ=0.5 멀티라벨 F1(micro/macro/sample).
    micro_f1이 모델 선택(metric_for_best_model) 기준. 벤더 연속성 앵커(top-1 weighted)도 병기.
    """
    logits, labels = eval_pred
    logits = np.asarray(logits)
    Y = np.asarray(labels).astype(int)
    pred = (_sigmoid(logits) >= 0.5).astype(int)
    return {
        "micro_f1":  f1_score(Y, pred, average="micro",   zero_division=0),   # headline (모델 선택 기준)
        "macro_f1":  f1_score(Y, pred, average="macro",   zero_division=0),
        "sample_f1": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),  # 벤더 연속성
    }

## 훈련

In [12]:
steps_per_epoch = math.ceil(len(dataset["train"]) / config["eff_batch"])
num = 4 if SEARCH else 2
eval_steps = math.ceil(steps_per_epoch / num)   # 탐색:에폭당 4회 / 풀런:에폭당 2회

training_args = TrainingArguments(
    output_dir='/workspace/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],   
    train_sampling_strategy="group_by_length",          # 유사 길이 배치로 padding 최소화
    remove_unused_columns=False,                        # 커스텀 collator가 키를 직접 선택 + length 컬럼 보존
    num_train_epochs=config["epochs"],
    bf16=True,                                          # ModernBERT 계열 안정성엔 bf16 (bf16=True, fp16=False)
    eval_strategy='steps',
    eval_steps=eval_steps,
    save_strategy="no" if SEARCH else "steps",          # 탐색:저장 안 함 / 풀런:에폭당 저장(볼륨)
    save_steps=eval_steps,                              # save_strategy="no"면 무시됨
    save_total_limit=3,
    logging_dir='/workspace/logs',
    logging_steps=50,
    metric_for_best_model="micro_f1",         
    greater_is_better=True,
    load_best_model_at_end=not SEARCH,                  # 풀런에서만 best 복원
    report_to="wandb",
    run_name=config["run_name"]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [14]:
## micro batch 확인 — worst-case 배치로 실제 스텝을 돌려 peak 측정
def probe_batches(model, vocab_size, seq_len, train_mb=(1, 2, 4), eval_mb=(2, 4, 8), num_labels=188):
    """
    worst case = 모든 시퀀스가 seq_len이고 mask가 전부 1(패딩 0).

    group_by_length가 시퀀스가 가장 큰 배치를 만들고 초반 할당한다.
    (random 샘플링이어도 배치에 장문 하나만 섞이면 배치 전체가 그 길이로 패딩된다).

    - AdamW state를 상주시킨 **정상상태** peak를 잰다.
      lr=0이라 가중치는 변하지 않는다 → 사전학습 가중치를 훼손하지 않음.
    - eval도 잰다: forward-only(no_grad)라 싸지만 eval 배치 역시 배치 내 최댓값으로 패딩되고,
      eval OOM은 첫 에폭 끝에서 런을 죽인다.
    """
    dev = model.device
    opt = torch.optim.AdamW(model.parameters(), lr=0.0)   # lr=0 → state만 할당, 가중치 불변

    def _mk(mb):
        ids = torch.randint(5, vocab_size, (mb, seq_len), device=dev)
        return ids, torch.ones_like(ids), torch.zeros(mb, num_labels, device=dev)

    model.train()
    for mb in train_mb:
        try:
            ids, mask, labels = _mk(mb)
            for step in (0, 1):                       # step0: AdamW state 할당 / step1: 정상상태 peak 측정
                if step == 1:
                    torch.cuda.reset_peak_memory_stats()
                with torch.autocast("cuda", dtype=torch.bfloat16):   # Trainer(bf16=True)와 동일 경로
                    out = model(input_ids=ids, attention_mask=mask)
                    loss = F.binary_cross_entropy_with_logits(out.logits.float(), labels)
                loss.backward()                                      # backward는 autocast 밖
                opt.step()
                opt.zero_grad(set_to_none=True)
            print(f"train micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"train micro={mb:>2}: OOM")
            opt.zero_grad(set_to_none=True)
            break                                     # 이후 후보는 자명하게 OOM
        finally:
            torch.cuda.empty_cache()

    model.eval()
    for mb in eval_mb:                                # 옵티마이저 state가 상주한 실제 조건에서 측정
        try:
            torch.cuda.reset_peak_memory_stats()
            ids, mask, _ = _mk(mb)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                model(input_ids=ids, attention_mask=mask)
            print(f"eval  micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"eval  micro={mb:>2}: OOM")
            break
        finally:
            torch.cuda.empty_cache()

    del opt                                           # Trainer가 자체 옵티마이저를 새로 만들도록 정리
    model.zero_grad(set_to_none=True)
    model.train()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()              # train() peak를 깨끗하게 재측정하기 위함


# 설정값 검증
probe_batches(
    model, tokenizer.vocab_size, config["max_len"],
    train_mb=(16, 32, 64, 96, 128, 160, 192, 224, 256),
    eval_mb=(32, 64, 128, 256, 512),
    num_labels=config["num_labels"],
)

train micro=16: peak   7.2 GB  OK
train micro=32: peak  12.2 GB  OK
train micro=64: peak  22.3 GB  OK
train micro=96: peak  32.5 GB  OK
train micro=128: peak  42.6 GB  OK
train micro=160: OOM
eval  micro=32: peak   2.5 GB  OK
eval  micro=64: peak   3.0 GB  OK
eval  micro=128: peak   3.9 GB  OK
eval  micro=256: peak   5.7 GB  OK
eval  micro=512: peak   9.3 GB  OK


In [14]:
import gc

# probe_batches가 옵티마이저·캐시를 내부에서 정리하므로 여기선 GC만 한 번 더 돌린다
gc.collect()
torch.cuda.empty_cache()
print(f"probe 후 잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

probe 후 잔여 allocated: 0.6 GB (모델 가중치)


In [15]:
trainer.train()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
395,0.003932,0.001763,0.089437,0.068130,0.052382,0.935227,0.397505
790,0.001303,0.000826,0.654015,0.613906,0.587328,0.273965,0.685468
1185,0.001041,0.000674,0.727492,0.702215,0.686191,0.173177,0.729471
1580,0.000870,0.000665,0.732708,0.706680,0.686660,0.193872,0.748467
1975,0.000733,0.000555,0.784444,0.773062,0.772205,0.084573,0.761512
2370,0.000657,0.000519,0.794356,0.783116,0.778542,0.095861,0.777195
2765,0.000629,0.000476,0.810052,0.801688,0.806266,0.062444,0.786137
3156,0.000565,0.000456,0.817033,0.809238,0.812928,0.059935,0.785205


TrainOutput(global_step=3156, training_loss=0.002113620811100714, metrics={'train_runtime': 5078.7728, 'train_samples_per_second': 79.505, 'train_steps_per_second': 0.621, 'total_flos': 1.2516793064335302e+17, 'train_loss': 0.002113620811100714, 'epoch': 2.0})

## 평가

In [16]:
if not SEARCH:
    test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
    for k, v in test_metrics.items():
        print(f"{k}: {v}")

In [17]:
if not SEARCH:
    os.makedirs(config["out_path"], exist_ok=True)
    trainer.save_model(config["out_path"])
    tokenizer.save_pretrained(config["out_path"])

    metrics_fp = os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json")
    with open(metrics_fp, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)

    print("saved", config["out_path"])

In [18]:
if not SEARCH:
    print(trainer.state.best_model_checkpoint)
    print(trainer.state.best_metric)

In [19]:
if not SEARCH:
    trainer.model.push_to_hub(config["repo_final"])
    tokenizer.push_to_hub(config["repo_final"])